# Module 2.2 — Synthetic Data Generation
**DeskMate SLM Curriculum · Phase 2 · Notebook 09**

Run on **Google Colab** or **Kaggle** (CPU is fine).

By the end you will have:
- A configurable prompt builder covering all 15 DeskMate intents
- A teacher generator using the Anthropic API (Path A) or Ollama (Path B)
- A quality-control pipeline: JSON parse → schema → span check → near-dup filter
- An augmented training set (`data/processed/train_augmented.jsonl`) with ≥30% real anchor
- A reusable `src/synth_generator.py` script

Read `2.2_synthetic_data.md` for theory and design rationale.

---
> **Cost note:** Path A (Anthropic API) costs ~$0.50–$2 for 5k examples.
> Path B (Ollama local) is free. The notebook auto-detects which to use.
---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q anthropic==0.34.0 sentence-transformers==3.1.1 jsonschema==4.23.0 pandas==2.2.2


In [ ]:
import random, json, re, time, pathlib, os
import pandas as pd
import numpy as np
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Ready.")


## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_PROC = PROJECT_ROOT / "data" / "processed"
SRC_DIR   = PROJECT_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f"Runtime : {RUNTIME}")
print(f"Src dir : {SRC_DIR}")


## Step 2 — Load Existing Real Training Data

In [ ]:
train_path = DATA_PROC / "train.jsonl"
if not train_path.exists():
    print("WARNING: train.jsonl not found — run Module 2.1 first.")
    print("Creating a minimal placeholder so the rest of this notebook can run.")
    placeholder = [
        {"text": "I can't log in to my account.", "intent": "account_access",
         "category": "account", "priority": "medium", "source": "placeholder"}
    ]
    train_path.write_text("\n".join(json.dumps(r) for r in placeholder))

real_df = pd.read_json(train_path, lines=True)
print(f"Real training examples: {len(real_df):,}")
print(f"Intent distribution (top 5):")
print(real_df["intent"].value_counts().head(5).to_string())


## Step 3 — Seed Configuration

Define cross-product seeds for maximum diversity:
product × version × error_code × scenario × persona.


In [ ]:
# DeskMate fictional product lineup
PRODUCTS = ["WorkflowAI", "ConnectHub", "DataSync Pro", "TaskBoard", "ReportKit"]
VERSIONS  = ["2.0.1", "3.1.0-beta", "4.5.2", "v1.9.3", "latest"]
ERROR_CODES = [
    "ERR_AUTH_TOKEN_EXPIRED", "ERR_RATE_LIMIT_429", "CONN_TIMEOUT_502",
    "EXPORT_FAILED_413", "SYNC_CONFLICT_409", "ERR_PERMISSION_DENIED",
    "WEBHOOK_DELIVERY_FAILED", "ERR_INVALID_API_KEY",
]

PERSONAS = [
    "frustrated enterprise IT administrator",
    "confused non-technical first-time user",
    "startup founder worried about data loss",
    "operations manager under tight deadline",
    "developer who discovered an edge-case bug",
]

# Per-intent scenario seeds (3–5 per intent drives variation)
INTENT_SEEDS = {
    "account_access":   ["forgot password", "2FA not working", "account locked after attempts",
                         "SSO redirect fails", "login works desktop not mobile"],
    "account_settings": ["need to update email", "change billing address",
                         "add team member", "rename workspace", "update timezone"],
    "billing_dispute":  ["charged twice", "unexpected charge", "wrong plan billed",
                         "invoice amount wrong", "coupon not applied"],
    "billing_inquiry":  ["what is included in plan", "when does trial end",
                         "how are seats counted", "invoice for accountant"],
    "cancellation":     ["want to cancel subscription", "cancel before renewal",
                         "pause subscription", "downgrade to free"],
    "data_privacy":     ["GDPR deletion request", "data breach concern",
                         "export all my data", "opt out of analytics",
                         "who has access to my data"],
    "feature_request":  ["dark mode please", "bulk export feature",
                         "API for webhooks", "custom domain support"],
    "onboarding":       ["how do I set up first project", "connect Slack integration",
                         "invite team members", "import from previous tool"],
    "outage_report":    ["service completely down", "dashboard not loading for 2 hours",
                         "all API calls failing", "status page shows green but broken"],
    "payment_failure":  ["credit card declined", "bank rejects payment",
                         "PayPal not accepted", "payment keeps failing"],
    "performance_issue":["dashboard loads very slowly", "search takes 30 seconds",
                         "app crashes on large project", "mobile app drains battery"],
    "refund_request":   ["request refund for unused months", "charged after cancellation",
                         "refund for wrong plan", "money back guarantee"],
    "technical_bug":    ["export button broken", "data not saving", "integration stopped working",
                         "webhook not triggering", "API returns 500"],
    "usage_question":   ["how do I export CSV", "where is the audit log",
                         "how to set permissions", "can I share a report link"],
    "out_of_scope":     ["looking for a job", "what is the weather",
                         "tell me a joke", "random unrelated query"],
}

total_combinations = sum(
    len(seeds) * len(PERSONAS) * (len(PRODUCTS) if intent not in ("out_of_scope",) else 1)
    for intent, seeds in INTENT_SEEDS.items()
)
print(f"Seed combinations available : {total_combinations:,}")
print(f"  (products × personas × seeds per intent)")


## Step 4 — Prompt Builder

In [ ]:
INTENTS_LIST = list(INTENT_SEEDS.keys())

def build_prompt(intent, product, version, error_code, scenario_seed, persona):
    fields_instruction = ""
    if intent in ("technical_bug", "performance_issue", "outage_report"):
        fields_instruction = (
            f"Include the product name '{product}', version '{version}', "
            f"and error code '{error_code}' naturally in the text. "
            "In the 'fields' object, include 'product', 'version', and 'error_code' "
            "with their exact character start/end positions in the text (0-indexed, end is exclusive)."
        )
    elif intent in ("billing_dispute", "billing_inquiry", "payment_failure", "refund_request"):
        fields_instruction = (
            f"Mention the product '{product}' naturally. "
            "In 'fields', include 'product' with its start/end positions."
        )
    else:
        fields_instruction = (
            "Leave 'fields' as an empty object {}."
        )

    return (
        "You are generating training data for a support ticket classifier.

"
        "Generate ONE realistic customer support ticket as a JSON object with EXACTLY these keys:
"
        '- "text": the ticket text (50-200 words, written by the persona described below)
'
        f'- "intent": must be exactly "{intent}"
'
        '- "category": the category for this intent (account/billing/data/product/technical/out_of_scope)
'
        '- "priority": must be one of "low", "medium", or "high"
'
        f'- "fields": {fields_instruction}

'
        f"Persona: {persona}
"
        f"Scenario seed: {scenario_seed}

"
        "Rules:
"
        "1. Write ONLY valid JSON - no markdown, no explanation, no preamble.
"
        "2. The text must sound authentic to the persona.
"
        "3. Priority must reflect actual business impact, not just urgency language.
"
        "4. Character spans in fields must exactly match text[start:end].

"
        "JSON:"
    )

# Test the prompt
sample = build_prompt(
    intent="technical_bug",
    product="WorkflowAI",
    version="3.1.0-beta",
    error_code="ERR_AUTH_TOKEN_EXPIRED",
    scenario_seed="export button broken",
    persona="frustrated enterprise IT administrator",
)
print(sample[:600] + "...")


## Step 5 — Detect Teacher Path (API vs Ollama)

The notebook auto-detects which teacher to use based on available secrets.
Set `FORCE_PATH` to override.


In [ ]:
FORCE_PATH = None   # set "api" or "ollama" to override auto-detection

def get_anthropic_key():
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
    try:
        import kaggle_secrets
        return kaggle_secrets.UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    except Exception:
        pass
    return os.environ.get("ANTHROPIC_API_KEY", "")

ANTHROPIC_KEY = get_anthropic_key()

if FORCE_PATH:
    TEACHER_PATH = FORCE_PATH
elif ANTHROPIC_KEY and len(ANTHROPIC_KEY) > 10:
    TEACHER_PATH = "api"
else:
    # Try Ollama
    try:
        import requests
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        TEACHER_PATH = "ollama" if r.ok else "mock"
    except Exception:
        TEACHER_PATH = "mock"

print(f"Teacher path: {TEACHER_PATH}")
if TEACHER_PATH == "mock":
    print("  No API key or Ollama found — using mock generator for demonstration.")
    print("  To use real generation: set ANTHROPIC_API_KEY secret OR start Ollama.")


## Step 6 — Generator Functions

In [ ]:
import anthropic as ant_lib

def generate_via_api(prompt, retries=3):
    client = ant_lib.Anthropic(api_key=ANTHROPIC_KEY)
    for attempt in range(retries):
        try:
            msg = client.messages.create(
                model="claude-haiku-20240307",
                max_tokens=512,
                messages=[{"role": "user", "content": prompt}],
            )
            return msg.content[0].text.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise e

def generate_via_ollama(prompt, model="llama3.1:8b"):
    import requests
    resp = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=120,
    )
    return resp.json()["response"].strip()

# Mock generator for offline demonstration
_MOCK_EXAMPLES = [
    {
        "text": "I upgraded WorkflowAI to v3.1.0-beta last week and now the export button returns ERR_AUTH_TOKEN_EXPIRED every time. Our whole team is blocked and we have a client delivery tomorrow.",
        "intent": "technical_bug", "category": "technical", "priority": "high",
        "fields": {
            "product":    {"value": "WorkflowAI", "start": 11, "end": 21},
            "version":    {"value": "v3.1.0-beta", "start": 25, "end": 36},
            "error_code": {"value": "ERR_AUTH_TOKEN_EXPIRED", "start": 82, "end": 104},
        }
    },
    {
        "text": "Hi, I was charged twice for my Pro plan last month. The invoice shows two identical charges of $49. Can you please refund the duplicate?",
        "intent": "billing_dispute", "category": "billing", "priority": "high",
        "fields": {"product": {"value": "Pro", "start": 39, "end": 42}}
    },
    {
        "text": "I can't log in. I reset my password three times but keep getting an error. I have a presentation in an hour.",
        "intent": "account_access", "category": "account", "priority": "medium",
        "fields": {}
    },
]
_mock_idx = [0]

def generate_mock(prompt):
    ex = _MOCK_EXAMPLES[_mock_idx[0] % len(_MOCK_EXAMPLES)]
    _mock_idx[0] += 1
    return json.dumps(ex)

def generate_one(prompt):
    if TEACHER_PATH == "api":
        return generate_via_api(prompt)
    elif TEACHER_PATH == "ollama":
        return generate_via_ollama(prompt)
    else:
        return generate_mock(prompt)

print(f"Generator ready (path: {TEACHER_PATH})")


## Step 7 — Quality Control Pipeline

In [ ]:
from jsonschema import validate, ValidationError

EXAMPLE_SCHEMA = {
    "type": "object",
    "required": ["text", "intent", "category", "priority", "fields"],
    "properties": {
        "text":     {"type": "string", "minLength": 10},
        "intent":   {"type": "string", "enum": INTENTS_LIST},
        "category": {"type": "string", "enum": ["account","billing","data","product","technical","out_of_scope"]},
        "priority": {"type": "string", "enum": ["low","medium","high"]},
        "fields":   {"type": "object"},
    },
    "additionalProperties": False,
}

PII_PATTERNS = [
    re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"),   # email
    re.compile(r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b"),           # credit card
    re.compile(r"\b\d{3}[-.\s]?\d{2}[-.\s]?\d{4}\b"),                       # SSN
]

def check_spans(example):
    text = example["text"]
    for field_name, field_val in example.get("fields", {}).items():
        if not isinstance(field_val, dict):
            return False, f"field '{field_name}' is not a dict"
        if "start" not in field_val or "end" not in field_val or "value" not in field_val:
            return False, f"field '{field_name}' missing start/end/value"
        s, e, v = field_val["start"], field_val["end"], field_val["value"]
        if not (0 <= s < e <= len(text)):
            return False, f"field '{field_name}' span [{s},{e}] out of range for text length {len(text)}"
        if text[s:e] != v:
            return False, f"field '{field_name}' span mismatch: text[{s}:{e}]={text[s:e]!r} != {v!r}"
    return True, "ok"

def validate_example(raw_text):
    stats = {"json": False, "schema": False, "spans": False, "pii": False}
    # 1. JSON parse
    try:
        # Extract JSON from possible surrounding text
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if not match:
            return None, stats
        obj = json.loads(match.group())
        stats["json"] = True
    except json.JSONDecodeError:
        return None, stats
    # 2. Schema
    try:
        validate(instance=obj, schema=EXAMPLE_SCHEMA)
        stats["schema"] = True
    except ValidationError:
        return None, stats
    # 3. Spans
    ok, msg = check_spans(obj)
    if not ok:
        return None, stats
    stats["spans"] = True
    # 4. PII
    for pattern in PII_PATTERNS:
        if pattern.search(obj["text"]):
            return None, stats
    stats["pii"] = True
    return obj, stats

# Quick test
test_raw = json.dumps(_MOCK_EXAMPLES[0])
result, qc_stats = validate_example(test_raw)
print(f"QC test: {qc_stats}")
print(f"Passed: {result is not None}")


## Step 8 — Generation Loop

In [ ]:
TARGET_PER_INTENT = 30 if TEACHER_PATH == "mock" else 100
# In production set to 300-500 per intent for a full synthetic set (~4500 examples)

print(f"Generating {TARGET_PER_INTENT} examples per intent ({len(INTENTS_LIST)} intents)")
print(f"Target total: {TARGET_PER_INTENT * len(INTENTS_LIST):,} examples (before QC drop)")
print()

synthetic_examples = []
qc_totals = {"json": 0, "schema": 0, "spans": 0, "pii": 0, "passed": 0, "generated": 0}

for intent in INTENTS_LIST:
    seeds    = INTENT_SEEDS[intent]
    passed   = 0
    attempts = 0
    max_attempts = TARGET_PER_INTENT * 3   # allow 3x calls to reach target after QC

    while passed < TARGET_PER_INTENT and attempts < max_attempts:
        # Sample a random combination
        product    = random.choice(PRODUCTS) if intent != "out_of_scope" else "N/A"
        version    = random.choice(VERSIONS) if intent not in ("out_of_scope","billing_inquiry","usage_question") else "N/A"
        error_code = random.choice(ERROR_CODES) if intent in ("technical_bug","outage_report","performance_issue") else "N/A"
        seed       = random.choice(seeds)
        persona    = random.choice(PERSONAS)

        prompt  = build_prompt(intent, product, version, error_code, seed, persona)
        raw     = generate_one(prompt)
        obj, stats = validate_example(raw)
        attempts += 1

        for k in stats:
            if stats[k]:
                qc_totals[k] += 1
        qc_totals["generated"] += 1

        if obj is not None:
            obj["source"] = "synthetic"
            synthetic_examples.append(obj)
            passed += 1

    print(f"  {intent:<25}: {passed:>4} / {TARGET_PER_INTENT}  (attempts: {attempts})")

qc_totals["passed"] = len(synthetic_examples)
print(f"\nTotal synthetic examples: {len(synthetic_examples):,}")
print(f"Pass rate: {len(synthetic_examples)/max(1,qc_totals['generated'])*100:.1f}%")


In [ ]:
# QC funnel summary
print("QC funnel:")
print(f"  Generated  : {qc_totals['generated']:>6}")
print(f"  Valid JSON : {qc_totals['json']:>6}  ({qc_totals['json']/max(1,qc_totals['generated'])*100:.1f}%)")
print(f"  Schema OK  : {qc_totals['schema']:>6}  ({qc_totals['schema']/max(1,qc_totals['generated'])*100:.1f}%)")
print(f"  Spans OK   : {qc_totals['spans']:>6}  ({qc_totals['spans']/max(1,qc_totals['generated'])*100:.1f}%)")
print(f"  No PII     : {qc_totals['pii']:>6}  ({qc_totals['pii']/max(1,qc_totals['generated'])*100:.1f}%)")
print(f"  Final kept : {qc_totals['passed']:>6}")


## Step 9 — Near-Dup Filter on Synthetic Set

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

synth_df = pd.DataFrame(synthetic_examples)
print(f"Before near-dup filter: {len(synth_df):,}")

if len(synth_df) > 10:
    enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embs = enc.encode(synth_df["text"].tolist(), batch_size=128,
                      convert_to_tensor=True, normalize_embeddings=True,
                      show_progress_bar=True)

    THRESHOLD = 0.92
    keep = torch.ones(len(synth_df), dtype=torch.bool)
    for i in range(len(synth_df)):
        if not keep[i]:
            continue
        sims = (embs[i] @ embs[i+1:].T)
        dups = (sims > THRESHOLD).nonzero(as_tuple=True)[0]
        for d in dups:
            keep[i+1+d.item()] = False

    synth_df = synth_df[keep.cpu().numpy()].reset_index(drop=True)

print(f"After near-dup filter : {len(synth_df):,}")


## Step 10 — Mix with Real Data & Save

In [ ]:
# Ensure real data columns match synthetic schema
# (source already present from Module 2.1; fields column may be absent)
if "fields" not in real_df.columns:
    real_df["fields"] = [{}] * len(real_df)

augmented_df = pd.concat([real_df, synth_df], ignore_index=True)

# Verify real-data anchor
real_fraction = len(real_df) / len(augmented_df)
print(f"Real examples    : {len(real_df):,}  ({real_fraction*100:.1f}%)")
print(f"Synthetic examples: {len(synth_df):,}  ({(1-real_fraction)*100:.1f}%)")
print(f"Total augmented  : {len(augmented_df):,}")
if real_fraction < 0.30:
    print("WARNING: real fraction < 30% — risk of model collapse. Generate fewer synthetic examples.")
else:
    print("Real-data anchor OK (>= 30%).")

# Shuffle
augmented_df = augmented_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

aug_path = DATA_PROC / "train_augmented.jsonl"
augmented_df.to_json(aug_path, orient="records", lines=True)
print(f"\nSaved: {aug_path}  ({aug_path.stat().st_size / 1024:.1f} KB)")


In [ ]:
# Intent distribution comparison: real-only vs augmented
print("Intent distribution comparison:")
print(f"{'Intent':<25} {'Real':>6} {'Aug':>6} {'Synth added':>12}")
print("-" * 52)
real_counts = real_df["intent"].value_counts()
aug_counts  = augmented_df["intent"].value_counts()
for intent in sorted(INTENTS_LIST):
    r = real_counts.get(intent, 0)
    a = aug_counts.get(intent, 0)
    print(f"{intent:<25} {r:>6} {a:>6} {a-r:>+12}")


## Step 11 — Save `src/synth_generator.py`

In [ ]:
script_lines = [
    '# synth_generator.py -- DeskMate synthetic ticket generator',
    '# Run: python synth_generator.py --intent technical_bug --n 50',
    'import argparse, json, os, random, re, time',
    'import anthropic',
    '',
    "PRODUCTS    = ['WorkflowAI', 'ConnectHub', 'DataSync Pro', 'TaskBoard', 'ReportKit']",
    "VERSIONS    = ['2.0.1', '3.1.0-beta', '4.5.2', 'v1.9.3', 'latest']",
    "ERROR_CODES = ['ERR_AUTH_TOKEN_EXPIRED','ERR_RATE_LIMIT_429','CONN_TIMEOUT_502',",
    "               'EXPORT_FAILED_413','SYNC_CONFLICT_409','ERR_PERMISSION_DENIED']",
    'PERSONAS = [',
    "    'frustrated enterprise IT administrator',",
    "    'confused first-time user',",
    "    'startup founder worried about data loss',",
    "    'operations manager under deadline',",
    "    'developer who found a bug',",
    ']',
    '',
    'def build_prompt(intent, product, version, error_code, seed, persona):',
    '    return (',
    "        'Generate a support ticket as JSON with keys: '",
    "        'text, intent, category, priority, fields. '",
    "        f'Intent must be {intent!r}. '",
    "        f'Persona: {persona}. Seed: {seed}. '",
    "        f'Product: {product} {version} error: {error_code}. '",
    "        'Output ONLY valid JSON, no preamble.'",
    '    )',
    '',
    'def generate(prompt, api_key):',
    '    client = anthropic.Anthropic(api_key=api_key)',
    '    msg = client.messages.create(',
    "        model='claude-haiku-20240307', max_tokens=512,",
    "        messages=[{'role': 'user', 'content': prompt}])",
    '    return msg.content[0].text.strip()',
    '',
    'def main():',
    '    ap = argparse.ArgumentParser()',
    "    ap.add_argument('--intent', required=True)",
    "    ap.add_argument('--n', type=int, default=50)",
    "    ap.add_argument('--out', default='synthetic_out.jsonl')",
    '    args = ap.parse_args()',
    "    api_key = os.environ.get('ANTHROPIC_API_KEY', '')",
    '    results = []',
    '    for _ in range(args.n * 3):',
    '        if len(results) >= args.n:',
    '            break',
    '        prompt = build_prompt(',
    '            args.intent, random.choice(PRODUCTS), random.choice(VERSIONS),',
    "            random.choice(ERROR_CODES), f'seed_{random.randint(1,100)}',",
    '            random.choice(PERSONAS))',
    '        try:',
    '            raw = generate(prompt, api_key)',
    "            m = re.search(r'{.*}', raw, re.DOTALL)",
    '            if m:',
    '                obj = json.loads(m.group())',
    "                obj['source'] = 'synthetic'",
    '                results.append(obj)',
    '        except Exception:',
    '            time.sleep(1)',
    "    with open(args.out, 'w') as f:",
    '        for r in results:',
    "            f.write(json.dumps(r) + '\\n')",
    "    print(f'Saved {len(results)} examples to {args.out}')",
    '',
    "if __name__ == '__main__':",
    '    main()'
]
gen_path = SRC_DIR / 'synth_generator.py'
gen_path.write_text('\n'.join(script_lines))
print(f'Saved: {gen_path}')
print('Usage: python src/synth_generator.py --intent technical_bug --n 100')


## Checkpoint

> **How do you keep synthetic data diverse, and why is a real-data anchor essential?**


In [ ]:
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Standalone generator script | `src/synth_generator.py` |
| Augmented training set | `data/processed/train_augmented.jsonl` |

**Next:** Module 2.3 — Data prep for encoder fine-tuning.
Tokenize the augmented dataset with the encoder's tokenizer, build attention masks,
encode labels as integers, and produce a `DatasetDict` ready for the `Trainer`.
